# Transformer Surprisal: GPT-2 & BERT

This notebook computes word-level surprisal from two transformer-based language models:

- **GPT-2** — autoregressive causal LM: surprisal = $-\log P(w_i \mid w_1 \ldots w_{i-1})$
- **BERT** — masked LM: surprisal approximated via pseudo-log-likelihood (PLL), masking each word and reading its logit

Both models are evaluated on:
- **Natural Stories** (primary dataset — self-paced reading times)
- **Dundee** (replication dataset — first-pass fixation duration)

At the end we produce a **4-model comparison table** (bigram / trigram / GPT-2 / BERT)
against human reading times to answer **RQ1** and **RQ2**.

In [ ]:
import os
import re
import sys
import math
import subprocess
import warnings
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

# ── Path resolver (mirrors notebook 2) ────────────────────────────────────────
def resolve_data_paths():
    """Resolve dataset locations for local and Kaggle environments."""
    kaggle_input = "/kaggle/input"

    if os.path.exists(kaggle_input):
        for root, dirs, _ in os.walk(kaggle_input):
            dirs_set = set(dirs)
            if "data" in dirs_set:
                data_root = os.path.join(root, "data")
                ns_candidate = os.path.join(data_root, "natural_stories")
                du_candidate = os.path.join(data_root, "dundee")
                if os.path.exists(ns_candidate) and os.path.exists(du_candidate):
                    return ns_candidate, du_candidate, "/kaggle/working/results", "kaggle"
            if "natural_stories" in dirs_set and "dundee" in dirs_set:
                ns_candidate = os.path.join(root, "natural_stories")
                du_candidate = os.path.join(root, "dundee")
                return ns_candidate, du_candidate, "/kaggle/working/results", "kaggle"

    # Local fallback
    script_dir = os.getcwd()
    project_root = os.path.abspath(os.path.join(script_dir, ".."))
    ns_local = os.path.join(project_root, "data", "natural_stories")
    du_local = os.path.join(project_root, "data", "dundee")
    results_local = os.path.join(project_root, "results")
    return ns_local, du_local, results_local, "local"

DATA_NS, DATA_DU, RESULTS, ENV = resolve_data_paths()
os.makedirs(RESULTS, exist_ok=True)
OUTPUTS_DIR = os.path.join(RESULTS, "notebook_outputs")
os.makedirs(OUTPUTS_DIR, exist_ok=True)

print(f"Environment     : {ENV}")
print(f"Natural Stories : {DATA_NS}")
print(f"Dundee          : {DATA_DU}")
print(f"Results         : {RESULTS}")

In [ ]:
# ── Install / import dependencies ──────────────────────────────────────────────
try:
    import torch
    from transformers import (
        GPT2LMHeadModel, GPT2TokenizerFast,
        BertForMaskedLM, BertTokenizerFast,
    )
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install",
                           "torch", "transformers", "--quiet"])
    import torch
    from transformers import (
        GPT2LMHeadModel, GPT2TokenizerFast,
        BertForMaskedLM, BertTokenizerFast,
    )

try:
    from scipy import stats
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "scipy", "--quiet"])
    from scipy import stats

import matplotlib
import matplotlib.pyplot as plt
matplotlib.rcParams.update({"font.size": 12, "figure.dpi": 120})

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# ── Utility helpers ────────────────────────────────────────────────────────────
def save_display_df(df, filename):
    path = os.path.join(OUTPUTS_DIR, filename)
    df.to_csv(path, index=False)
    print(f"Saved display output: {path}")

def save_text(text, filename):
    path = os.path.join(OUTPUTS_DIR, filename)
    with open(path, "w") as f:
        f.write(text)
    print(f"Saved text output: {path}")

def normalize_token(token):
    """Same normalizer used in notebook 2 for consistent matching."""
    if pd.isna(token):
        return "<unk>"
    token = str(token).strip().lower()
    token = re.sub(r"[^a-z0-9'\-]+", "", token)
    return token if token else "<unk>"

---
## Part 1: GPT-2 Surprisal
---

In [ ]:
# ── 1.1 Load GPT-2 model and tokenizer ────────────────────────────────────────
GPT2_MODEL = "gpt2"   # 117M params; upgrade to gpt2-medium if desired

print(f"Loading {GPT2_MODEL} ...")
gpt2_tokenizer = GPT2TokenizerFast.from_pretrained(GPT2_MODEL)
gpt2_model     = GPT2LMHeadModel.from_pretrained(GPT2_MODEL).to(DEVICE)
gpt2_model.eval()

GPT2_MAX_TOKENS = gpt2_model.config.n_positions  # 1024 for gpt2
# We use half as a safe chunk size so there is headroom
GPT2_CHUNK_TOKENS = GPT2_MAX_TOKENS // 2         # 512

print(f"GPT-2 loaded. Max positions: {GPT2_MAX_TOKENS}, chunk size: {GPT2_CHUNK_TOKENS} tokens")

In [ ]:
# ── 1.2 GPT-2 surprisal function  (chunked to respect max 1024-token limit) ───
#
# FIX: GPT-2 positional embeddings only go up to 1024 tokens.
# A story with 400+ words easily produces 700-1500+ BPE tokens, blowing the
# limit and causing a CUDA device-side assertion (cudaErrorAssert).
# Solution: split the word list into non-overlapping chunks of <= 512 tokens;
# compute surprisal within each chunk independently.
# The first word in each chunk gets NaN (no cross-chunk context) — this is
# standard practice and has negligible effect on corpus-level correlations.

@torch.no_grad()
def gpt2_word_surprisals(words):
    """
    Compute GPT-2 surprisal (bits) for each word in a sequence.
    Handles texts of arbitrary length by processing in chunks of
    GPT2_CHUNK_TOKENS subword tokens.
    """
    if not words:
        return []

    # Step 1: Tokenize each word independently to get a word → [token_ids] map.
    # A leading space is added from word 1 onward so GPT-2 treats them as
    # mid-sentence words (BPE boundary handling).
    word_token_ids = []
    for i, w in enumerate(words):
        text = (" " if i > 0 else "") + (str(w).strip() if w else "")
        ids  = gpt2_tokenizer.encode(text, add_special_tokens=False)
        if not ids:                              # empty word → placeholder
            ids = [gpt2_tokenizer.unk_token_id]
        word_token_ids.append(ids)

    surprisals = [float("nan")] * len(words)    # initialise all to NaN

    # Step 2: Walk through words, accumulating a chunk that stays <= CHUNK_TOKENS.
    chunk_start = 0
    while chunk_start < len(words):

        # Greedily add words until we would exceed the token budget
        chunk_word_indices = []
        token_budget       = GPT2_CHUNK_TOKENS
        wi                 = chunk_start
        while wi < len(words) and len(word_token_ids[wi]) <= token_budget:
            chunk_word_indices.append(wi)
            token_budget -= len(word_token_ids[wi])
            wi += 1

        # Edge case: a single word has more tokens than the budget (very rare)
        if not chunk_word_indices:
            chunk_word_indices = [wi]
            wi += 1

        # Build flat token sequence for this chunk
        flat_ids = [
            tid
            for idx in chunk_word_indices
            for tid in word_token_ids[idx]
        ]

        # Safety clamp — should not be needed, but avoids silent CUDA crashes
        flat_ids   = flat_ids[:GPT2_MAX_TOKENS]
        input_ids  = torch.tensor([flat_ids], dtype=torch.long, device=DEVICE)

        # One forward pass
        logits     = gpt2_model(input_ids).logits[0]          # (seq_len, vocab)
        log_probs  = torch.log_softmax(logits, dim=-1).cpu()  # keep on CPU for indexing

        # token j's conditional log-prob comes from the distribution at position j-1
        token_log_probs = []
        for j, tid in enumerate(flat_ids):
            if j == 0:
                token_log_probs.append(float("nan"))  # first token: no context
            else:
                token_log_probs.append(log_probs[j - 1, tid].item())

        # Aggregate subword token log-probs → word-level surprisal
        pos = 0
        for local_i, global_i in enumerate(chunk_word_indices):
            ids = word_token_ids[global_i]
            n   = min(len(ids), len(token_log_probs) - pos)  # clamp for safety
            sub = token_log_probs[pos : pos + n]
            if not sub or any(math.isnan(v) for v in sub):
                surprisals[global_i] = float("nan")
            else:
                word_log_prob        = sum(sub)              # sum in log-space
                surprisals[global_i] = -word_log_prob / math.log(2)  # nats → bits
            pos += n

        chunk_start = wi   # advance to the next unprocessed word

    return surprisals


# Sanity check on a short sentence
test  = ["The", "cat", "sat", "on", "the", "mat"]
tsurp = gpt2_word_surprisals(test)
print("GPT-2 sanity check (bits):")
for w, s in zip(test, tsurp):
    print(f"  {w:12s}  {'NaN (first / no context)' if math.isnan(s) else f'{s:.3f}'}")

In [ ]:
# ── 1.3 Compute GPT-2 surprisal — Natural Stories ─────────────────────────────
ns_words = pd.read_csv(os.path.join(DATA_NS, "ns_words.csv"))

ns_gpt2_parts = []
for story_id, grp in tqdm(ns_words.groupby("story"), desc="NS GPT-2"):
    grp         = grp.sort_values("zone").copy()
    words_list  = grp["word_text"].fillna("").tolist()
    surp_values = gpt2_word_surprisals(words_list)
    grp["gpt2_surprisal"] = surp_values
    ns_gpt2_parts.append(grp)

ns_gpt2     = pd.concat(ns_gpt2_parts, ignore_index=True)
ns_gpt2_out = os.path.join(RESULTS, "ns_gpt2_surprisal.csv")
ns_gpt2[["story", "zone", "word_text", "gpt2_surprisal"]].to_csv(ns_gpt2_out, index=False)

print(f"Saved: {ns_gpt2_out}")
print(f"Rows : {len(ns_gpt2)}")
desc = ns_gpt2["gpt2_surprisal"].describe().round(4)
print("\nGPT-2 surprisal summary (Natural Stories):")
print(desc.to_string())
save_display_df(desc.reset_index(), "ns_gpt2_describe.csv")

In [ ]:
# ── 1.4 Compute GPT-2 surprisal — Dundee ──────────────────────────────────────
du_words = pd.read_csv(os.path.join(DATA_DU, "dundee_word_agg.csv"))

du_gpt2_parts = []
for text_id, grp in tqdm(du_words.groupby("text_num"), desc="Dundee GPT-2"):
    grp         = grp.sort_values("WNUM").copy()
    words_list  = grp["word_text"].fillna("").tolist()
    surp_values = gpt2_word_surprisals(words_list)
    grp["gpt2_surprisal"] = surp_values
    du_gpt2_parts.append(grp)

du_gpt2     = pd.concat(du_gpt2_parts, ignore_index=True)
du_gpt2_out = os.path.join(RESULTS, "dundee_gpt2_surprisal.csv")
du_gpt2[["text_num", "WNUM", "word_text", "gpt2_surprisal"]].to_csv(du_gpt2_out, index=False)

print(f"Saved: {du_gpt2_out}")
print(f"Rows : {len(du_gpt2)}")
desc = du_gpt2["gpt2_surprisal"].describe().round(4)
print("\nGPT-2 surprisal summary (Dundee):")
print(desc.to_string())
save_display_df(desc.reset_index(), "du_gpt2_describe.csv")

---
## Part 2: BERT Pseudo-Log-Likelihood (PLL) Surprisal
---

In [ ]:
# ── 2.1 Load BERT model and tokenizer ─────────────────────────────────────────
BERT_MODEL = "bert-base-uncased"

print(f"Loading {BERT_MODEL} ...")
bert_tokenizer = BertTokenizerFast.from_pretrained(BERT_MODEL)
bert_model     = BertForMaskedLM.from_pretrained(BERT_MODEL).to(DEVICE)
bert_model.eval()

BERT_MAX_TOKENS  = bert_tokenizer.model_max_length   # 512 for bert-base
# Safe number of *words* per chunk (each word ≈ 1-3 sub-tokens on average)
BERT_CHUNK_WORDS = 80   # ≈ 80-160 sub-tokens + 2 special tokens → well under 512

print(f"BERT loaded. Max tokens: {BERT_MAX_TOKENS}, chunk word size: {BERT_CHUNK_WORDS}")

In [ ]:
# ── 2.2 BERT PLL surprisal function  (chunked to respect max 512-token limit) ─
#
# FIX: BERT's positional embeddings are capped at 512 tokens.  Long texts
# (e.g. a Dundee article with 2000+ words ≈ 3000+ subword tokens) silently
# truncate, causing the second half of the text to be scored against a
# truncated context — or, if truncation fills all 512 slots with the first
# N words, all remaining words get NaN.
# Solution: process the word list in non-overlapping chunks of BERT_CHUNK_WORDS.

@torch.no_grad()
def bert_word_surprisals(words):
    """
    Compute BERT PLL surprisal (bits) for each word.

    For each word:
      - Mask all its subword tokens simultaneously
      - One BERT forward pass
      - Sum log P(subword | masked context) across the word's pieces
      - Negate and convert to bits

    Long texts are split into chunks of BERT_CHUNK_WORDS words so the
    subword sequence always fits within 512 positions.
    """
    if not words:
        return []

    surprisals = [float("nan")] * len(words)
    MASK_ID     = bert_tokenizer.mask_token_id

    # Process the word list in non-overlapping word-chunks
    for chunk_start in range(0, len(words), BERT_CHUNK_WORDS):
        chunk_words = words[chunk_start : chunk_start + BERT_CHUNK_WORDS]
        chunk_words = [str(w).strip() if w else "" for w in chunk_words]

        # Tokenize the whole chunk as a pre-split word list
        encoding  = bert_tokenizer(
            chunk_words,
            is_split_into_words = True,
            return_tensors      = "pt",
            truncation          = True,
            max_length          = BERT_MAX_TOKENS,
            padding             = False,
        )

        chunk_input_ids = encoding["input_ids"].to(DEVICE)    # (1, seq_len)
        chunk_attn_mask = encoding["attention_mask"].to(DEVICE)
        word_ids        = encoding.word_ids(batch_index=0)    # None for [CLS]/[SEP]

        # Map each word-in-chunk index to its token positions
        n_chunk_words = len(chunk_words)
        w2tok = {i: [] for i in range(n_chunk_words)}
        for tok_pos, wid in enumerate(word_ids):
            if wid is not None:
                w2tok[wid].append(tok_pos)

        # Score each word in this chunk
        for local_wi in range(n_chunk_words):
            global_wi  = chunk_start + local_wi
            tok_positions = w2tok.get(local_wi, [])
            if not tok_positions:
                continue   # word was truncated away — leave as NaN

            # Clone and mask all subword positions for this word
            masked_ids = chunk_input_ids.clone()
            for tp in tok_positions:
                masked_ids[0, tp] = MASK_ID

            logits = bert_model(
                masked_ids, attention_mask=chunk_attn_mask
            ).logits   # (1, seq_len, vocab)

            log_prob_sum = 0.0
            for tp in tok_positions:
                true_id       = chunk_input_ids[0, tp].item()
                lp            = torch.log_softmax(logits[0, tp], dim=-1)[true_id].item()
                log_prob_sum += lp

            surprisals[global_wi] = -log_prob_sum / math.log(2)   # nats → bits

    return surprisals


# Sanity check
test  = ["The", "cat", "sat", "on", "the", "mat"]
tsurp = bert_word_surprisals(test)
print("BERT PLL sanity check (bits):")
for w, s in zip(test, tsurp):
    print(f"  {w:12s}  {'NaN' if math.isnan(s) else f'{s:.3f}'}")

In [ ]:
# ── 2.3 Compute BERT PLL — Natural Stories ─────────────────────────────────────
# On a Kaggle T4 GPU:  ~5-10 min total for Natural Stories (~10K words)

ns_bert_parts = []
for story_id, grp in tqdm(ns_words.groupby("story"), desc="NS BERT"):
    grp         = grp.sort_values("zone").copy()
    words_list  = grp["word_text"].fillna("").tolist()
    surp_values = bert_word_surprisals(words_list)
    grp["bert_surprisal"] = surp_values
    ns_bert_parts.append(grp)

ns_bert     = pd.concat(ns_bert_parts, ignore_index=True)
ns_bert_out = os.path.join(RESULTS, "ns_bert_surprisal.csv")
ns_bert[["story", "zone", "word_text", "bert_surprisal"]].to_csv(ns_bert_out, index=False)

print(f"Saved: {ns_bert_out}")
print(f"Rows : {len(ns_bert)}")
desc = ns_bert["bert_surprisal"].describe().round(4)
print("\nBERT PLL surprisal summary (Natural Stories):")
print(desc.to_string())
save_display_df(desc.reset_index(), "ns_bert_describe.csv")

In [ ]:
# ── 2.4 Compute BERT PLL — Dundee ─────────────────────────────────────────────
# Dundee ~50K words → ~30-40 min on CPU, ~8-15 min on T4 GPU

du_bert_parts = []
for text_id, grp in tqdm(du_words.groupby("text_num"), desc="Dundee BERT"):
    grp         = grp.sort_values("WNUM").copy()
    words_list  = grp["word_text"].fillna("").tolist()
    surp_values = bert_word_surprisals(words_list)
    grp["bert_surprisal"] = surp_values
    du_bert_parts.append(grp)

du_bert     = pd.concat(du_bert_parts, ignore_index=True)
du_bert_out = os.path.join(RESULTS, "dundee_bert_surprisal.csv")
du_bert[["text_num", "WNUM", "word_text", "bert_surprisal"]].to_csv(du_bert_out, index=False)

print(f"Saved: {du_bert_out}")
print(f"Rows : {len(du_bert)}")
desc = du_bert["bert_surprisal"].describe().round(4)
print("\nBERT PLL surprisal summary (Dundee):")
print(desc.to_string())
save_display_df(desc.reset_index(), "du_bert_describe.csv")

---
## Part 3: 4-Model Comparison — Surprisal vs Human Reading Time
---

In [ ]:
# ── 3.1 Load n-gram surprisal results from notebook 2 ──────────────────────────
ns_ngram = pd.read_csv(os.path.join(RESULTS, "ns_ngram_surprisal.csv"))
du_ngram = pd.read_csv(os.path.join(RESULTS, "dundee_ngram_surprisal.csv"))

ns_agg   = pd.read_csv(os.path.join(DATA_NS, "ns_word_agg.csv"))
du_agg   = pd.read_csv(os.path.join(DATA_DU, "dundee_word_agg.csv"))

print("N-gram NS rows :", len(ns_ngram))
print("N-gram DU rows :", len(du_ngram))
print("NS  agg rows   :", len(ns_agg))
print("DU  agg rows   :", len(du_agg))

In [ ]:
# ── 3.2 Build merged evaluation dataframes ─────────────────────────────────────
ns_eval = (
    ns_agg
    .merge(ns_ngram[["story", "zone", "bigram_surprisal", "trigram_surprisal"]],
           on=["story", "zone"], how="inner")
    .merge(ns_gpt2[["story", "zone", "gpt2_surprisal"]],
           on=["story", "zone"], how="inner")
    .merge(ns_bert[["story", "zone", "bert_surprisal"]],
           on=["story", "zone"], how="inner")
)

du_eval = (
    du_agg
    .merge(du_ngram[["text_num", "WNUM", "bigram_surprisal", "trigram_surprisal"]],
           on=["text_num", "WNUM"], how="inner")
    .merge(du_gpt2[["text_num", "WNUM", "gpt2_surprisal"]],
           on=["text_num", "WNUM"], how="inner")
    .merge(du_bert[["text_num", "WNUM", "bert_surprisal"]],
           on=["text_num", "WNUM"], how="inner")
)

print(f"NS  eval rows: {len(ns_eval):,}")
print(f"DU  eval rows: {len(du_eval):,}")

In [ ]:
# ── 3.3 Compute Pearson correlations ──────────────────────────────────────────
def pearson_r(df, surp_col, dv_col):
    """Return Pearson r and p-value, dropping NaNs first."""
    valid = df[[surp_col, dv_col]].dropna()
    r, p  = stats.pearsonr(valid[surp_col], valid[dv_col])
    return round(r, 4), round(p, 6)

models = [
    ("Bigram KN",  "bigram_surprisal"),
    ("Trigram KN", "trigram_surprisal"),
    ("GPT-2",      "gpt2_surprisal"),
    ("BERT (PLL)", "bert_surprisal"),
]

rows = []
for model_name, col in models:
    ns_r, ns_p = pearson_r(ns_eval, col, "mean_log_RT")
    du_r, du_p = pearson_r(du_eval, col, "mean_log_FDUR")
    rows.append({
        "Model":              model_name,
        "NS r(logRT)":        ns_r,
        "NS p":               ns_p,
        "Dundee r(logFDUR)": du_r,
        "Dundee p":           du_p,
    })

corr_table = pd.DataFrame(rows)
print("=" * 70)
print("4-Model Surprisal Correlation with Human Reading Times")
print("=" * 70)
print(corr_table.to_string(index=False))

corr_table.to_csv(os.path.join(RESULTS, "all_models_correlation_summary.csv"), index=False)
save_display_df(corr_table, "all_models_correlation_table.csv")
save_text(corr_table.to_string(index=False), "all_models_correlation_summary.txt")

In [ ]:
# ── 3.4 Visualize: correlation bar chart ──────────────────────────────────────
model_names = corr_table["Model"].tolist()
ns_rs       = corr_table["NS r(logRT)"].tolist()
du_rs       = corr_table["Dundee r(logFDUR)"].tolist()

x     = list(range(len(model_names)))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
bars1 = ax.bar([xi - width/2 for xi in x], ns_rs, width,
               label="Natural Stories (logRT)",  color="steelblue", alpha=0.85)
bars2 = ax.bar([xi + width/2 for xi in x], du_rs, width,
               label="Dundee (logFDUR)",          color="coral",     alpha=0.85)

for bar in list(bars1) + list(bars2):
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width() / 2., h + 0.003,
            f"{h:.3f}", ha="center", va="bottom", fontsize=10)

ax.axhline(0, color="black", linewidth=0.8)
ax.set_ylabel("Pearson r")
ax.set_title("Surprisal–Reading Time Correlation by Model (RQ1 & RQ2)")
ax.set_xticks(x)
ax.set_xticklabels(model_names)
ax.legend()
all_rs = ns_rs + du_rs
ax.set_ylim(min(all_rs) - 0.05, max(all_rs) + 0.08)

plt.tight_layout()
fig_dir = os.path.join(RESULTS, "report_figures")
os.makedirs(fig_dir, exist_ok=True)
fig_path = os.path.join(fig_dir, "fig_model_comparison.png")
plt.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved figure: {fig_path}")

In [ ]:
# ── 3.5 Scatter plots: surprisal vs log reading time (all models × corpora) ───
fig, axes = plt.subplots(2, 4, figsize=(20, 9))

datasets = [
    (ns_eval, "mean_log_RT",   "Natural Stories", "steelblue"),
    (du_eval, "mean_log_FDUR", "Dundee",          "coral"),
]

for row_idx, (df, dv_col, ds_name, color) in enumerate(datasets):
    for col_idx, (model_name, surp_col) in enumerate(models):
        ax  = axes[row_idx, col_idx]
        sub = df[[surp_col, dv_col]].dropna()

        ax.scatter(sub[surp_col], sub[dv_col],
                   alpha=0.15, s=4, color=color, rasterized=True)

        m, b = np.polyfit(sub[surp_col], sub[dv_col], 1)
        xs   = np.linspace(sub[surp_col].min(), sub[surp_col].max(), 200)
        ax.plot(xs, m * xs + b, color="black", linewidth=1.5)

        r, _ = pearson_r(df, surp_col, dv_col)
        ax.set_title(f"{model_name}\n{ds_name}  r={r}", fontsize=10)
        ax.set_xlabel("Surprisal (bits)", fontsize=9)
        ax.set_ylabel("log DV", fontsize=9)

plt.suptitle("Surprisal vs Log Reading Time — All Models & Corpora", fontsize=13)
plt.tight_layout()
scatter_path = os.path.join(fig_dir, "fig_scatter_all_models.png")
plt.savefig(scatter_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved figure: {scatter_path}")

In [ ]:
# ── 3.6 Surprisal distribution histograms ─────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (df, title) in zip(axes, [
    (ns_eval, "Natural Stories"),
    (du_eval, "Dundee"),
]):
    for model_name, col in models:
        vals = df[col].dropna()
        ax.hist(vals, bins=80, density=True, alpha=0.5, label=model_name)
    ax.set_xlabel("Surprisal (bits)")
    ax.set_ylabel("Density")
    ax.set_title(f"Surprisal Distributions — {title}")
    ax.legend(fontsize=9)

plt.tight_layout()
dist_path = os.path.join(fig_dir, "fig_surp_distributions_all.png")
plt.savefig(dist_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved figure: {dist_path}")

In [ ]:
# ── 3.7 Print answers to RQ1 and RQ2 ─────────────────────────────────────────
print("=" * 60)
print("RESULTS SUMMARY")
print("=" * 60)
print()
print("RQ1: Does LM surprisal correlate with human reading time?")
print("-" * 60)
for _, row in corr_table.iterrows():
    print(f"  {row['Model']:15s} | NS r={row['NS r(logRT)']:+.4f}  "
          f"Dundee r={row['Dundee r(logFDUR)']:+.4f}")
print()
print("RQ2: Do transformer LMs outperform n-gram LMs?")
print("-" * 60)
kn_rows  = corr_table[corr_table["Model"].str.contains("KN")]
tr_rows  = corr_table[corr_table["Model"].str.contains("GPT|BERT")]
ngram_ns_best  = kn_rows["NS r(logRT)"].max()
transf_ns_best = tr_rows["NS r(logRT)"].max()
ngram_du_best  = kn_rows["Dundee r(logFDUR)"].max()
transf_du_best = tr_rows["Dundee r(logFDUR)"].max()
print(f"  Best n-gram     NS={ngram_ns_best:+.4f}  Dundee={ngram_du_best:+.4f}")
print(f"  Best transformer NS={transf_ns_best:+.4f}  Dundee={transf_du_best:+.4f}")
verdict_ns = "Transformers OUTPERFORM n-grams" if transf_ns_best > ngram_ns_best else "N-grams match/outperform transformers"
verdict_du = "Transformers OUTPERFORM n-grams" if transf_du_best > ngram_du_best else "N-grams match/outperform transformers"
print(f"  Natural Stories → {verdict_ns}")
print(f"  Dundee          → {verdict_du}")